In [1]:
# Used to Hide Codes-----------------

from IPython.display import HTML

HTML(r"""
<style>
  #code-toggle-btn {
      position: fixed;
      top: 10px;
      right: 10px;
      z-index: 9999;
      padding: 6px 12px;
      border-radius: 4px;
      border: 1px solid #999;
      background: #f5f5f5;
      cursor: pointer;
      font-family: inherit;
      font-size: 13px;
  }
</style>

<script>
  var codeVisible = false;

  function toggleCode() {
      if (codeVisible) {
          $('div.input').hide();
          $('#code-toggle-btn').text('Show code');
      } else {
          $('div.input').show();
          $('#code-toggle-btn').text('Hide code');
      }
      codeVisible = !codeVisible;
  }

  // Run when notebook page is ready
  $(document).ready(function () {
      // Hide all code cells initially
      $('div.input').hide();
      codeVisible = false;

      // Add the floating button
      if (!document.getElementById('code-toggle-btn')) {
          $('body').append(
              '<button id="code-toggle-btn" onclick="toggleCode()">Show code</button>'
          );
      }
  });
</script>
""")


In [2]:
#use to display in separate window
#%matplotlib qt

In [3]:

#  INSTALLS  (run once in a notebook / terminal)

# !pip install pandas numpy matplotlib seaborn
# !pip install ipywidgets
# !pip install praw
# !pip install nltk
# !pip install textblob vaderSentiment
# !pip install pymongo
# !pip install wordcloud
# !pip install pyspark
# !pip install tqdm
# !pip install transformers torch hf_xet
#!pip install transformers torch vaderSentiment

#!pip install hf_xet
# import nltk
# nltk.download("punkt")
# nltk.download("stopwords")

In [4]:
#  IMPORTS
import os
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import praw
from pymongo import MongoClient

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


In [5]:
#Configurations for Batch Collections  

#  Load private settings from the .env file (never upload .env to GitHub)
from dotenv import load_dotenv
load_dotenv()

#  Project Path (set CYBER_PROJECT_DIR in .env, or files are saved to ./data)
PROJECT = os.getenv("CYBER_PROJECT_DIR", os.path.join(os.getcwd(), "data"))
os.makedirs(PROJECT, exist_ok=True)

BATCH = os.path.join(PROJECT, "reddit_cybersecurity_batch.csv")
CLEAN = os.path.join(PROJECT, "cybersecurity_clean.csv")
SENT  = os.path.join(PROJECT, "cybersecurity_sentiment.csv")
LIVE  = os.path.join(PROJECT, "live_stream")
os.makedirs(LIVE, exist_ok=True)

#  MONGODB CONFIG

mongo_client = MongoClient(os.getenv("MONGO_URI", "mongodb://localhost:27017/"))
mongo_db     = mongo_client[os.getenv("MONGO_DB_NAME", "cyber_stream")]

#  REDDIT CONFIG (values come from .env)

R_CLIENT_ID     = os.getenv("REDDIT_CLIENT_ID")
R_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET")
R_USER_AGENT    = os.getenv("REDDIT_USER_AGENT", "CyberStreamCoursework/1.0")

if not R_CLIENT_ID or not R_CLIENT_SECRET:
    raise RuntimeError("Missing Reddit credentials. Copy .env.example to .env and fill it in.")

reddit = praw.Reddit(
    client_id=R_CLIENT_ID,
    client_secret=R_CLIENT_SECRET,
    user_agent=R_USER_AGENT,
)

# Cybersecurity-related subreddits
subs = [
    "cybersecurity", "netsec", "privacy", "infosec", "hacking",
    "malware", "blueteamsec", "netsecstudents", "pwned",
    "dataisugly", "infosecNews", "AskNetsec", "ReverseEngineering",
]


#  HELPERS: STOPWORDS & VADER
stop_words = set(stopwords.words("english"))
vader      = SentimentIntensityAnalyzer()



In [6]:
# BATCH DATA COLLECTION FROM REDDIT

def fetch_batch(limit_per_sub=1500):
    rows = []
    for s in subs:
        print(f"Fetching from r/{s}...")
        try:
            posts = reddit.subreddit(s).new(limit=limit_per_sub)
            for post in posts:
                rows.append({
                    "id":         post.id,
                    "subreddit":  s.lower(),
                    "title":      post.title or "",
                    "text":       post.selftext or "",
                    "created_utc": post.created_utc,
                    "url":        post.url,
                })
        except Exception as e:
            print(f"  Error fetching r/{s}: {e}")
            time.sleep(1)
    df = pd.DataFrame(rows)
    print("Fetched rows:", len(df))
    return df

def save_csv(df, filename):
    if df.empty:
        print(f"CSV NOT saved – DataFrame empty ({filename}).")
        return
    df.to_csv(filename, index=False, encoding="utf-8")
    print(f"CSV saved: {filename} ({len(df)} rows)")

def save_mongodb(df, collection_name):
    if df.empty:
        print(f"MongoDB NOT saved – DataFrame empty ({collection_name}).")
        return
    col     = mongo_db[collection_name]
    records = df.to_dict(orient="records")
    res     = col.insert_many(records, ordered=False)
    print(f"MongoDB saved: {len(res.inserted_ids)} docs → {mongo_db.name}.{collection_name}")

#  RUN BATCH COLLECTION (comment out if you already have the CSV) 
df_batch = fetch_batch(limit_per_sub=1500)
save_csv(df_batch, BATCH)
save_mongodb(df_batch, "reddit_batch")


Fetching from r/cybersecurity...
Fetching from r/netsec...
Fetching from r/privacy...
Fetching from r/infosec...
Fetching from r/hacking...
Fetching from r/malware...
Fetching from r/blueteamsec...
Fetching from r/netsecstudents...
Fetching from r/pwned...
Fetching from r/dataisugly...
Fetching from r/infosecNews...
Fetching from r/AskNetsec...
Fetching from r/ReverseEngineering...
Fetched rows: 12523
CSV saved: data\reddit_cybersecurity_batch.csv (12523 rows)
MongoDB saved: 12523 docs → cyber_stream.reddit_batch


In [7]:
#  CLEANING & PREPARATION

def light_clean(text):
  
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    return text.lower().strip()

def clean_text(text):
    
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and w.isalpha()]
    return " ".join(tokens).strip()

df_clean = pd.read_csv(BATCH)
print("Raw rows:", len(df_clean))

df_clean["title"] = df_clean["title"].fillna("")
df_clean["text"]  = df_clean["text"].fillna("")
df_clean["content"] = (df_clean["title"] + " " + df_clean["text"]).str.strip()

# remove empty + super short content
df_clean = df_clean[df_clean["content"].str.len() > 0].copy()
df_clean["word_count"] = df_clean["content"].str.split().str.len()
df_clean = df_clean[df_clean["word_count"] >= 4].copy()

# add cleaned versions
df_clean["content_lc"] = df_clean["content"].apply(light_clean)
df_clean["clean_text"] = df_clean["content"].apply(clean_text)
df_clean = df_clean[df_clean["clean_text"].str.len() > 0].copy()
df_clean.drop(columns=["word_count"], inplace=True)

print("Rows after cleaning:", len(df_clean))
df_clean.to_csv(CLEAN, index=False, encoding="utf-8")
print(f"{len(df_clean)} rows → {CLEAN}")
save_mongodb(df_clean, "reddit_clean")

Raw rows: 12523
Rows after cleaning: 12235
12235 rows → data\cybersecurity_clean.csv
MongoDB saved: 12235 docs → cyber_stream.reddit_clean


In [8]:
#  Sentiment set up for  TEXTBLOB + VADER

def get_tb_score(text):
    return float(TextBlob(text).sentiment.polarity)

def get_tb_label(score):
    if score > 0.1:
        return "positive"
    elif score < -0.1:
        return "negative"
    else:
        return "neutral"

def get_vader_score(text):
    if not isinstance(text, str):
        text = ""
    return float(vader.polarity_scores(text)["compound"])

def get_vader_label(score):
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    else:
        return "neutral"

df_sent = df_clean.copy()
text_source = "content_lc"          # recommended

df_sent["tb_score"]    = df_sent[text_source].apply(get_tb_score)
df_sent["tb_label"]    = df_sent["tb_score"].apply(get_tb_label)
df_sent["vader_score"] = df_sent[text_source].apply(get_vader_score)
df_sent["vader_label"] = df_sent["vader_score"].apply(get_vader_label)

df_sent.to_csv(SENT, index=False, encoding="utf-8")
print(f"(SENT) saved {len(df_sent)} rows → {SENT}")
save_mongodb(df_sent, "reddit_sentiment")


#    Sentiment Set up for RoBERTa (CardiffNLP)

roberta_model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
model     = AutoModelForSequenceClassification.from_pretrained(roberta_model_name)
labels_roberta = ["negative", "neutral", "positive"]

def get_roberta_label_score(text):
    
    if not isinstance(text, str) or not text.strip():
        return "neutral", 0.0
    try:
        encoded = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding=True,
        )
        with torch.no_grad():
            output = model(**encoded)
            scores = output.logits[0].detach().numpy()
            probs  = np.exp(scores) / np.sum(np.exp(scores))
            label_id = int(np.argmax(probs))
        return labels_roberta[label_id], float(probs[label_id])
    except Exception:
        return "neutral", 0.0

print("Running RoBERTa sentiment on all posts... (this may take a while)")
roberta_results = df_sent[text_source].apply(get_roberta_label_score)
df_sent["roberta_label"] = roberta_results.apply(lambda x: x[0])
df_sent["roberta_score"] = roberta_results.apply(lambda x: x[1])
print("Done RoBERTa sentiment.")


#   SUMMARY COUNTS (THREE MODELS)

labels = ["positive", "neutral", "negative"]

tb_counts      = df_sent["tb_label"].value_counts().reindex(labels, fill_value=0)
vader_counts   = df_sent["vader_label"].value_counts().reindex(labels, fill_value=0)
roberta_counts = df_sent["roberta_label"].value_counts().reindex(labels, fill_value=0)

print("TextBlob counts:\n", tb_counts)
print("\nVADER counts:\n", vader_counts)
print("\nRoBERTa counts:\n", roberta_counts)


(SENT) saved 12235 rows → data\cybersecurity_sentiment.csv
MongoDB saved: 12235 docs → cyber_stream.reddit_sentiment
Running RoBERTa sentiment on all posts... (this may take a while)
Done RoBERTa sentiment.
TextBlob counts:
 tb_label
positive    3929
neutral     7161
negative    1145
Name: count, dtype: int64

VADER counts:
 vader_label
positive    5186
neutral     3237
negative    3812
Name: count, dtype: int64

RoBERTa counts:
 roberta_label
positive    1160
neutral     6975
negative    4100
Name: count, dtype: int64


In [9]:
#  HELPER: VALUE LABELS INSIDE BARS

def add_value_labels_inside(ax, fmt="{:d}", fontsize=11):
    for p in ax.patches:
        height = p.get_height()
        if height == 0:
            continue
        ax.text(
            p.get_x() + p.get_width() / 2,
            height / 2,
            fmt.format(int(height)),
            va="center",
            ha="center",
            color="white",
            fontsize=fontsize,
            fontweight="bold",
        )

In [13]:
# Plothings -------------

#  GRAPH 1: OVERALL COMPARISON (TEXTBLOB vs VADER vs RoBERTa)

fig, ax = plt.subplots(figsize=(10, 6))

x      = np.arange(len(labels))
width  = 0.25

bars_tb      = ax.bar(x - width,     tb_counts.values,      width=width, label="TextBlob",     edgecolor="black")
bars_vader   = ax.bar(x,             vader_counts.values,   width=width, label="VADER",        edgecolor="black")
bars_roberta = ax.bar(x + width, roberta_counts.values, width=width, label="RoBERTa",          edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Number of posts")
ax.set_title("Overall Sentiment Comparison: TextBlob vs VADER vs RoBERTa")
ax.legend()

add_value_labels_inside(ax, fontsize=9)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()


#  GRAPH 2: THREE PIE CHARTS (ONE PER MODEL)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].pie(tb_counts.values, labels=labels, autopct="%1.1f%%", startangle=90)
axes[0].set_title("TextBlob Sentiment Share")

axes[1].pie(vader_counts.values, labels=labels, autopct="%1.1f%%", startangle=90)
axes[1].set_title("VADER Sentiment Share")

axes[2].pie(roberta_counts.values, labels=labels, autopct="%1.1f%%", startangle=90)
axes[2].set_title("RoBERTa Sentiment Share")

plt.tight_layout()
plt.show()


#  GRAPH 3: VADER SENTIMENT BY SUBREDDIT

if "subreddit" not in df_sent.columns:
    raise KeyError("Column 'subreddit' not found in df_sent.")

vader_sub = (
    df_sent
    .groupby(["subreddit", "vader_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=labels, fill_value=0)
)

fig, ax = plt.subplots(figsize=(20, 10))
subreddits = vader_sub.index
sentiments = vader_sub.columns
n_groups   = len(subreddits)
n_sent     = len(sentiments)

bar_width    = 1.5
group_spacing = 0.45
x = np.arange(n_groups) * (n_sent * bar_width + group_spacing)
colors = ["steelblue", "darkorange", "darkcyan"]

for i, sentiment in enumerate(sentiments):
    ax.bar(
        x + i * bar_width,
        vader_sub[sentiment],
        width=bar_width,
        label=sentiment.capitalize(),
        edgecolor="black",
        linewidth=0.8,
        color=colors[i],
    )

ax.set_xticks(x + (n_sent - 1) * bar_width / 2)
ax.set_xticklabels(subreddits, rotation=45, ha="right")

ax.set_title("VADER Sentiment by Subreddit", fontsize=18, pad=15)
ax.set_xlabel("Subreddit", fontsize=16)
ax.set_ylabel("Number of posts", fontsize=16)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.legend(title="Sentiment", frameon=False, fontsize=12, title_fontsize=12, loc="upper right")

add_value_labels_inside(ax, fontsize=8)
plt.tight_layout()
plt.show()


#  GRAPH 4: RoBERTa SENTIMENT BY SUBREDDIT

roberta_sub = (
    df_sent
    .groupby(["subreddit", "roberta_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=labels, fill_value=0)
)

fig, ax = plt.subplots(figsize=(20, 10))
subreddits = roberta_sub.index
sentiments = roberta_sub.columns
n_groups   = len(subreddits)
n_sent     = len(sentiments)

bar_width    = 1.5
group_spacing = 0.45
x = np.arange(n_groups) * (n_sent * bar_width + group_spacing)
colors = ["mediumpurple", "slategray", "seagreen"]

for i, sentiment in enumerate(sentiments):
    ax.bar(
        x + i * bar_width,
        roberta_sub[sentiment],
        width=bar_width,
        label=sentiment.capitalize(),
        edgecolor="black",
        linewidth=0.8,
        color=colors[i],
    )

ax.set_xticks(x + (n_sent - 1) * bar_width / 2)
ax.set_xticklabels(subreddits, rotation=45, ha="right")

ax.set_title("RoBERTa Sentiment by Subreddit", fontsize=18, pad=15)
ax.set_xlabel("Subreddit", fontsize=16)
ax.set_ylabel("Number of posts", fontsize=16)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.legend(title="Sentiment", frameon=False, fontsize=12, title_fontsize=12, loc="upper right")

add_value_labels_inside(ax, fontsize=8)
plt.tight_layout()
plt.show()
